# [Step 4] EDA 반영 전처리 및 파생 피쳐(Feature) 생성


In [ ]:
# =====================================================================
# [코랩 전용] EDA 반영 전처리 및 4대 영역 파생 피쳐(Feature) 생성 파이프라인
# =====================================================================

import os
from pathlib import Path
import pandas as pd
import numpy as np

# 1. 경로 정의 및 폴더 생성 확인
BASE_DIR = Path("/content/drive/MyDrive/datamining_project")
DATA_RAW = BASE_DIR / "data" / "raw"
DATA_PROCESSED = BASE_DIR / "data" / "processed"
DATA_PROCESSED.mkdir(parents=True, exist_ok=True)

def build_anomaly_features_pipeline(file_path, output_name):
    print(f"\n⚙️ [{output_name.upper()} 데이터셋] 전처리 및 파생 피쳐 빌드 가동...")

    # 데이터 로드
    df = pd.read_csv(file_path)
    df["date"] = pd.to_datetime(df["date"])

    # -------------------------------------------------------------
    # [1] 전처리: 연산 에러 방지 (무한대 및 비활성 거래일 정제)
    # -------------------------------------------------------------
    # 수익률 변동 계산 시 분모가 0이 되어 발생하는 inf 값을 NaN으로 치환
    df.replace([np.inf, -np.inf], np.nan, inplace=True)

    # 거래량이 0인 날(거래정지 등)은 뒤에서 ZeroDivisionError를 유발하므로 결측화
    df["volume"] = df["volume"].replace(0, np.nan)

    # 주가 분석의 핵심 필수 컬럼 중 결측치가 있는 행 제거
    df.dropna(subset=["open", "high", "low", "close", "volume"], inplace=True)

    # -------------------------------------------------------------
    # [2] 전처리: EDA 반영 결과 구현 (K-Means 거리 왜곡 방지 클리핑)
    # -------------------------------------------------------------
    # EDA 7에서 확인한 극단적인 아웃라이어 폭발이 군집 중심을 파괴하지 않도록
    # 금융 퀀트 표준인 1% 및 99% 분위수 스케일 컷(Clipping)을 적용합니다.
    for col in ["close", "volume"]:
        lower_bound = df[col].quantile(0.01)
        upper_bound = df[col].quantile(0.99)
        df[col] = df[col].clip(lower_bound, upper_bound)

    # 종목별, 날짜별 시계열 순서 정렬 (윈도우 함수 연산 뒤틀림 방지)
    df = df.sort_values(["ticker", "date"]).copy()

    # -------------------------------------------------------------
    # [3] 피쳐 엔지니어링: 4대 핵심 영역 파생 변수 빌드
    # -------------------------------------------------------------

    # ── 영역 1. 거래량 과열 지표 (Volume Dynamics) ──
    df["prev_volume"] = df.groupby("ticker")["volume"].shift(1)
    # 전날 대비 거래량 변화율 (EDA 4, 7에서 검증한 핵심 지표)
    df["vol_chg_rate"] = (df["volume"] - df["prev_volume"]) / df["prev_volume"]
    # 최근 20일 평균 거래량 대비 당일 거래량 배수 (일시적 거래량 폭발 포착)
    df["volume_ma20_ratio"] = df["volume"] / df.groupby("ticker")["volume"].transform(lambda x: x.rolling(20).mean())

    # ── 영역 2. 가격 변동성 지표 (Price & Volatility) ──
    df["prev_close"] = df.groupby("ticker")["close"].shift(1)
    df["daily_return"] = (df["close"] - df["prev_close"]) / df["prev_close"]
    # 최근 5일간의 수익률 표준편차 (단기 가격 요동성 포착 - EDA 3 기반)
    df["volatility_5d"] = df.groupby("ticker")["daily_return"].transform(lambda x: x.rolling(5).std())
    # 최근 5일 고점 대비 현재가 낙폭 (급등 후 급락하는 패턴 포착)
    rolling_high_5d = df.groupby("ticker")["high"].transform(lambda x: x.rolling(5).max())
    df["drawdown_after_peak_5d"] = (rolling_high_5d - df["close"]) / rolling_high_5d

    # ── 영역 3. 캔들 마이크로 구조 지표 (Candle Microstructure) ──
    candle_range = (df["high"] - df["low"]).replace(0, np.nan)
    # 윗꼬리 비율: 당일 최고점 대비 시가/종가 중 높은 가격의 위치 (매물 출회 및 설거지 신호 - EDA 5 기반)
    df["upper_shadow_ratio"] = (df["high"] - df[["open", "close"]].max(axis=1)) / candle_range
    # 몸통 비율: 캔들 전체 크기 중 순수 시가-종가 몸통이 차지하는 비중
    df["body_ratio"] = (df["close"] - df["open"]).abs() / candle_range

    # ── 영역 4. 시계열 지속성 지표 (Pattern Streaks) ──
    # 윗꼬리 비율이 0.4(40%) 이상으로 길게 달린 비정상 거래일 여부를 임시 플래그(1)로 지정
    df["is_upper_shadow"] = (df["upper_shadow_ratio"] > 0.4).astype(int)
    # 최근 5일 중 윗꼬리가 달린 날의 누적 횟수 (비정상적인 누르기 및 분배 패턴의 지속성 포착)
    df["upper_shadow_streak_5d"] = df.groupby("ticker")["is_upper_shadow"].transform(lambda x: x.rolling(5).sum())

    # -------------------------------------------------------------
    # [4] 후처리: 윈도우 연산으로 발생한 초기 결측치 제거 및 저장
    # -------------------------------------------------------------
    # 20일 이동평균 등을 계산하느라 앞부분에 생긴 NaN 행들을 깔끔하게 제거합니다.
    essential_features = ["volume_ma20_ratio", "volatility_5d", "upper_shadow_streak_5d"]
    df.dropna(subset=essential_features, inplace=True)

    # 연산용 임시 컬럼 드랍
    df.drop(columns=["prev_volume", "prev_close", "is_upper_shadow"], errors="ignore", inplace=True)

    # 최종 결과물 저장
    save_path = DATA_PROCESSED / f"{output_name}_features.csv"
    df.to_csv(save_path, index=False, encoding="utf-8-sig")

    print(f"💾 [{output_name.upper()}] 피쳐셋 생성 완료! ➡️ 데이터 차원: {df.shape}")
    print(f"   • 생성된 마스터 피쳐: vol_chg_rate, volume_ma20_ratio, volatility_5d, drawdown_after_peak_5d, upper_shadow_ratio, body_ratio, upper_shadow_streak_5d")

    return df

# 분할된 3개 데이터셋에 파이프라인 일괄 적용
train_features = build_anomaly_features_pipeline(DATA_RAW / "train.csv", "train")
valid_features = build_anomaly_features_pipeline(DATA_RAW / "valid.csv", "valid")
test_features  = build_anomaly_features_pipeline(DATA_RAW / "test.csv", "test")

print("\n" + "="*60)
print("🎉 [성공] 전처리 및 4대 영역 핵심 파생 피쳐 적재 완료!")
print("="*60)


⚙️ [TRAIN 데이터셋] 전처리 및 파생 피쳐 빌드 가동...
💾 [TRAIN] 피쳐셋 생성 완료! ➡️ 데이터 차원: (42400, 17)
   • 생성된 마스터 피쳐: vol_chg_rate, volume_ma20_ratio, volatility_5d, drawdown_after_peak_5d, upper_shadow_ratio, body_ratio, upper_shadow_streak_5d

⚙️ [VALID 데이터셋] 전처리 및 파생 피쳐 빌드 가동...
💾 [VALID] 피쳐셋 생성 완료! ➡️ 데이터 차원: (3807, 17)
   • 생성된 마스터 피쳐: vol_chg_rate, volume_ma20_ratio, volatility_5d, drawdown_after_peak_5d, upper_shadow_ratio, body_ratio, upper_shadow_streak_5d

⚙️ [TEST 데이터셋] 전처리 및 파생 피쳐 빌드 가동...
💾 [TEST] 피쳐셋 생성 완료! ➡️ 데이터 차원: (3701, 17)
   • 생성된 마스터 피쳐: vol_chg_rate, volume_ma20_ratio, volatility_5d, drawdown_after_peak_5d, upper_shadow_ratio, body_ratio, upper_shadow_streak_5d

🎉 [성공] 전처리 및 4대 영역 핵심 파생 피쳐 적재 완료!


In [ ]:
# 전처리 단계

# 1. 무한대 값(inf , -inf) 제거 및 NaN 치환

# 왜 하냐? 전날 대비 거래량 변화율 (vol_chg_rate) 나 당일 수익률 계산할때
# 분모가 0으로 수렴할 경우 무한대 값(inf)가 발생합니다.
# inf 값을 NaN으로 통일하고 Drop

# 2. 거래량 0인 날 결측 처리 및 드랍

# 왜 하냐? 특정 종목이 거래가 아예 단절된 날이 있으면 거래량이 0으로 찍힙니다.
# 이러면 저희가 관찰을 할 수 없기때문에 전처리 과정을 거쳐야합니다.
# 시장의 유효한 매매활동이 존재했던 '활성 영업일' 만 볼 수 있게 데이터를 정제하여
# 모델이 무의미한 구간을 오인하는 오탐지율을 획기적으로 낮춥니다.



In [ ]:
# 피쳐 엔지니어링 단계
# 영역 1 거래량 과열 지표
# - 대형주와 소형주 하루 거래량 차이는 많이 납니다.
# - 종목 고유의 평소 거래대금 대비 상대적으로 얼마나 대량 거래가 터졌는지 봐야됩니다

# 영역 2 가격 변동성 지표
# - 소형주는 구조적으로 가격 변동성 꼬리가 깁니다.
# - 최근 5일간 얼마나 주가가 움직였는지 낙폭이 얼마나 심한지 포착합니다.
# - 단발성 주가 상승이 아닌 인위적인 패턴까지 다차원적으로 검출합니다.

# 영역 3 캔들 마이크로 구조 지표
# - 비정상 거래일에는 종가만 보면 안되고 장중의 움직임을 봐야됩니다.
# - 이상 패턴 신호를 포착하게 만듭니다.

# 영역 4 시계열 지속성 지표
# - 어쩌다 하루 이상할 수 있지만 5일 중 2회, 3회 연속된다면 이상하다고 판단합니다.
# - 최종 k-means 군집화 시 행동 유형별로 완성도 높게 클러스터링 되도록합니다.